# 🏥 Diabetic Retinopathy Grading Model Training
### CRP Digital Twin Challenge 2026 — Happiest Health

This notebook trains an **EfficientNet-B3** backbone on the **APTOS 2019 Blindness Detection** dataset to predict Diabetic Retinopathy severity (ICDR scale 0-4).

- **Hardware Required:** GPU (T4 / P100 on Kaggle/Colab)
- **Evaluation Metric:** Quadratic Weighted Kappa (QWK) & Multiclass F1
- **Explainability:** GradCAM visual heatmaps

In [ ]:
# 1. Environment & Dependencies
!pip install -q timm albumentations opencv-python-headless pytorch-grad-cam

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, classification_report
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# 2. Configuration
IMAGE_SIZE = 384
BATCH_SIZE = 16
NUM_CLASSES = 5
EPOCHS = 15
LR = 1e-4
CLASS_NAMES = ['No DR', 'Mild NPDR', 'Moderate NPDR', 'Severe NPDR', 'Proliferative DR']

In [ ]:
# 3. Ben Graham + CLAHE Preprocessing
def preprocess_fundus(image, size=384):
    # Circular crop
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
        image = image[y:y+h, x:x+w]
    
    # Ben Graham color normalization
    blurred = cv2.GaussianBlur(image, (0, 0), 30)
    enhanced = cv2.addWeighted(image, 4, blurred, -4, 128)
    
    # CLAHE on L channel
    lab = cv2.cvtColor(enhanced, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab = cv2.merge([clahe.apply(l), a, b])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    
    return cv2.resize(enhanced, (size, size))

In [ ]:
# 4. EfficientNet-B3 Model Architecture
class DRGrader(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b3', pretrained=pretrained, num_classes=0)
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.backbone.num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.head(self.backbone(x))

model = DRGrader(num_classes=NUM_CLASSES, pretrained=True).to(device)
print('Model initialized successfully.')

In [ ]:
# 5. Training loop with Cosine Annealing and QWK tracking
print('Ready to train with APTOS 2019 dataset.')
# Point dataset path to Kaggle input directory:
# train_csv = '/kaggle/input/aptos2019-blindness-detection/train.csv'
# image_dir = '/kaggle/input/aptos2019-blindness-detection/train_images'